# ORGANVM reader-mode documentation estate audit

**Audit date:** 2026-08-31

**Scope:** 323 repositories visible through the linked GitHub context

**Decision:** preserve one canonical factual substrate and expose it through
class-appropriate reader routes; repair truth and authority drift before
expanding search surfaces.

## TL;DR

This notebook normalizes seven independently produced repository inventories,
validates their coverage and scoring, publishes aggregate estate statistics,
and emits a privacy-safe public rollout queue. Private repository names and
findings remain outside the committed artifacts.

## Context & Methods

Each repository was inspected read-only and classified A–F. Seven dimensions
were scored from 0–4: orientation, technical depth, conceptual depth,
commercial relevance, evidence, search surface, and cross-linking. The seven
audit segments use slightly different field names, so this notebook maps them
into one schema and then asserts:

- exactly 323 rows and 323 unique `owner/repository` keys;
- classes are limited to A–F;
- visibility is public or private;
- all seven dimension scores fall within 0–4;
- recomputed totals agree with the normalized dimensions.

The ranking is editorial, not a claim of mathematical optimality. It combines
documentation class, public leverage, evidence/truth repair, and the explicit
recommendations recorded by each audit segment. Wave 0 establishes authority;
five pilots stress distinct rhetorical modes; the following twenty are the
first conversion queue.

### Key assumptions

- Repository visibility and metadata reflect the authorized exports captured
  for the 2026-08-31 audit, not a continuously refreshed GitHub view.
- Each repository is the unit of analysis; all seven rubric dimensions carry
  equal weight and use the source auditors' integer scores.
- A 0–28 total is descriptive, not a quality ranking. Conversion order also
  considers documentation class, truth/identity risk, public leverage, and
  segment recommendations.
- The seven raw exports are transient, access-controlled inputs. They are
  intentionally absent from this public repository; the committed manifest
  records filenames, counts, and hashes, not row-level private data.

## Data

The input grain is one repository per row across seven non-overlapping audit
segments. The next cells load the authorized exports, normalize their field
aliases, and validate coverage, visibility, classification, score completeness,
and source totals before any public artifact is written.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 100)

DIMENSIONS = (
    "orientation",
    "technical_depth",
    "conceptual_depth",
    "commercial_relevance",
    "evidence",
    "seo_surface",
    "cross_linking",
)
SCORE_ALIASES = {
    "orientation": ("orientation",),
    "technical_depth": ("technical_depth", "technical"),
    "conceptual_depth": ("conceptual_depth", "conceptual"),
    "commercial_relevance": ("commercial_relevance", "commercial"),
    "evidence": ("evidence",),
    "seo_surface": ("seo_surface", "seo"),
    "cross_linking": ("cross_linking",),
}
SOURCE_FILES = {
    "personal": "personal.json",
    "ergon": "ergon.json",
    "theoria_poiesis": "theoria-poiesis.json",
    "governance_comms": "governance-comms.json",
    "umbrella_core": "umbrella-core.json",
    "umbrella_extended": "umbrella-extended.json",
    "organvm_gap": "organvm-gap.json",
}

INPUT_DIR = Path(
    os.environ.get("ORGANVM_DOC_AUDIT_INPUT_DIR", "../../../audit")
).resolve()
OUTPUT_DIR = Path.cwd().resolve()
manifest_path = OUTPUT_DIR / "reader-mode-input-manifest.json"
pinned_input_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if pinned_input_manifest.get("schema_version") != "reader-mode-input-manifest.v1":
    raise ValueError("Pinned input manifest has an unsupported schema version")
pinned_sources = pinned_input_manifest.get("sources")
if not isinstance(pinned_sources, list):
    raise TypeError("Pinned input manifest sources must be a list")
pinned_by_segment = {
    item["source_segment"]: item
    for item in pinned_sources
    if isinstance(item, dict) and isinstance(item.get("source_segment"), str)
}
if len(pinned_by_segment) != len(pinned_sources):
    raise ValueError("Pinned input manifest has invalid or duplicate source segments")
if set(pinned_by_segment) != set(SOURCE_FILES):
    raise ValueError("Pinned input manifest source segments do not match the notebook")


def manifest_visibility(row, source, index):
    visibility = row.get("visibility")
    if visibility in {"public", "private"}:
        return visibility
    if visibility is not None:
        raise ValueError(
            f"{source} row {index} has invalid visibility: {visibility!r}"
        )
    public = row.get("metadata", {}).get("public")
    if isinstance(public, bool):
        return "public" if public else "private"
    raise ValueError(f"{source} row {index} has no valid visibility")


raw = {}
live_input_sources = []
for source, filename in SOURCE_FILES.items():
    expected = pinned_by_segment[source]
    if expected.get("filename") != filename:
        raise ValueError(f"Pinned filename mismatch for source segment {source}")
    source_path = INPUT_DIR / filename
    source_bytes = source_path.read_bytes()
    bundle = json.loads(source_bytes)
    repositories = bundle.get("repositories") if isinstance(bundle, dict) else None
    if not isinstance(repositories, list):
        raise TypeError(f"Authorized input {filename} has no repositories list")
    visibility_counts = {"public": 0, "private": 0}
    for index, row in enumerate(repositories):
        if not isinstance(row, dict):
            raise TypeError(f"{source} row {index} is not an object")
        visibility_counts[manifest_visibility(row, source, index)] += 1
    actual = {
        "source_segment": source,
        "filename": filename,
        "rows": len(repositories),
        "public": visibility_counts["public"],
        "private": visibility_counts["private"],
        "sha256": hashlib.sha256(source_bytes).hexdigest(),
    }
    if actual != expected:
        raise ValueError(
            f"Authorized input {filename} does not match its pinned bytes/counts"
        )
    raw[source] = bundle
    live_input_sources.append(actual)

live_input_totals = {
    "source_segments": len(live_input_sources),
    "repositories": sum(item["rows"] for item in live_input_sources),
    "public": sum(item["public"] for item in live_input_sources),
    "private": sum(item["private"] for item in live_input_sources),
}
if pinned_input_manifest.get("totals") != live_input_totals:
    raise ValueError("Pinned input manifest totals do not match the live inputs")
source_counts = {
    source: len(bundle["repositories"])
    for source, bundle in raw.items()
}
source_count_display = {
    SOURCE_FILES[source]: count for source, count in source_counts.items()
}
display(
    pd.DataFrame.from_dict(
        source_count_display,
        orient="index",
        columns=["repository rows"],
    )
)

,repository rows
personal.json,24
ergon.json,52
theoria-poiesis.json,30
governance-comms.json,47
umbrella-core.json,51
umbrella-extended.json,49
organvm-gap.json,70


In [2]:
def normalize_visibility(row):
    visibility = row.get("visibility")
    if visibility in {"public", "private"}:
        return visibility
    if visibility is not None:
        raise ValueError(
            f"Invalid visibility for {row.get('repository')}: {visibility!r}"
        )
    public = row.get("metadata", {}).get("public")
    if isinstance(public, bool):
        return "public" if public else "private"
    raise ValueError(f"Cannot resolve visibility for {row.get('repository')}")


def normalize_repository(source, bundle, row):
    if SOURCE_FILES[source] == "personal.json":
        return f"{bundle.get('owner', '4444J99')}/{row['name']}"
    return row["repository"]


def require_integer(value, field, repository):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(
            f"{repository}: {field} must be an integer, got {type(value).__name__}"
        )
    return value


def normalize_scores(row, repository):
    source_scores = row.get("scores", {})
    result = {}
    for dimension, aliases in SCORE_ALIASES.items():
        present = [key for key in aliases if source_scores.get(key) is not None]
        if len(present) != 1:
            raise ValueError(
                f"{repository}: expected exactly one score alias for {dimension}; "
                f"found {present}"
            )
        source_key = present[0]
        result[dimension] = require_integer(
            source_scores[source_key], f"scores.{source_key}", repository
        )

    total_candidates = [
        ("scores.total", source_scores.get("total")),
        ("score_total", row.get("score_total")),
        ("total", row.get("total")),
    ]
    present_totals = [
        (field, require_integer(value, field, repository))
        for field, value in total_candidates
        if value is not None
    ]
    if not present_totals:
        raise ValueError(f"{repository}: source total is required for this audit")
    if len({value for _, value in present_totals}) != 1:
        raise ValueError(
            f"{repository}: conflicting source totals in "
            f"{[field for field, _ in present_totals]}"
        )

    result["source_total"] = present_totals[0][1]
    result["total"] = sum(result[dimension] for dimension in DIMENSIONS)
    if result["source_total"] != result["total"]:
        raise ValueError(
            f"{repository}: source total {result['source_total']} does not equal "
            f"recomputed total {result['total']}"
        )
    return result


def normalize_readme(source, row):
    if SOURCE_FILES[source] == "personal.json":
        return bool(row.get("readme", {}).get("present"))
    if SOURCE_FILES[source] == "ergon.json":
        return bool(row.get("readme_observation", {}).get("fetched"))
    if SOURCE_FILES[source] == "theoria-poiesis.json":
        return int(row.get("readme_words", 0)) > 0
    if SOURCE_FILES[source] == "umbrella-extended.json":
        return bool(row.get("readme", {}).get("present"))
    if SOURCE_FILES[source] == "organvm-gap.json":
        return bool(row.get("readme", {}).get("present"))
    return None


def normalize_row(source, bundle, row):
    repository = normalize_repository(source, bundle, row)
    scores = normalize_scores(row, repository)
    metadata = row.get("metadata", {})
    finding = (
        row.get("finding")
        or row.get("note")
        or row.get("observed")
        or row.get("evidence_bounded_note")
        or "; ".join(row.get("truth_or_identity_findings", []))
        or ""
    )
    role = (
        row.get("repository_role")
        or row.get("surface_role")
        or row.get("role")
        or row.get("surface", {}).get("role")
        or "unspecified"
    )
    priority = (
        row.get("conversion_priority")
        or row.get("conversion_wave")
        or row.get("priority")
        or "unranked"
    )
    archived = bool(row.get("archived", metadata.get("archived", False)))
    return {
        "repository": repository,
        "source_segment": source,
        "visibility": normalize_visibility(row),
        "documentation_class": row.get("documentation_class") or row.get("class"),
        "repository_role": role,
        "priority": priority,
        "archived": archived,
        "has_readme": normalize_readme(source, row),
        **scores,
        "finding": finding,
        "recommended_treatment": row.get("recommended_treatment", ""),
        "recommended_reader_modes": row.get("recommended_reader_modes", []),
        "industry_clusters": row.get("industry_clusters", []),
        "concept_clusters": row.get("concept_clusters", []),
        "topics": metadata.get("topics", []),
        "url": row.get("url") or f"https://github.com/{repository}",
    }


def require(condition, message):
    if not condition:
        raise ValueError(message)


rows = [
    normalize_row(source, bundle, row)
    for source, bundle in raw.items()
    for row in bundle["repositories"]
]
inventory = pd.DataFrame(rows).sort_values("repository").reset_index(drop=True)

require(len(inventory) == 323, f"expected 323 rows, found {len(inventory)}")
require(inventory["repository"].nunique() == 323, "repository keys are not unique")
require(
    len(SOURCE_FILES) == 7,
    f"expected seven source segments, found {len(SOURCE_FILES)}",
)
require(inventory["source_segment"].nunique() == 7, "source segment coverage drifted")
require(
    set(inventory["documentation_class"]) <= set("ABCDEF"),
    "documentation classes must be limited to A-F",
)
require(
    set(inventory["visibility"]) <= {"public", "private"},
    "visibility values must be public or private",
)
for dimension in DIMENSIONS:
    require(
        inventory[dimension].between(0, 4).all(),
        f"{dimension} contains an out-of-range score",
    )
require(
    (inventory[list(DIMENSIONS)].sum(axis=1) == inventory["total"]).all(),
    "normalized totals do not equal the seven dimensions",
)
require(
    (inventory["source_total"] == inventory["total"]).all(),
    "source totals do not equal recomputed totals",
)
visibility_validation = inventory["visibility"].value_counts().to_dict()
require(
    visibility_validation == {"public": 239, "private": 84},
    f"visibility counts drifted: {visibility_validation}",
)

input_sources = []
for source, filename in SOURCE_FILES.items():
    source_rows = inventory[inventory["source_segment"] == source]
    source_visibility = source_rows["visibility"].value_counts().to_dict()
    source_path = INPUT_DIR / filename
    input_sources.append(
        {
            "source_segment": source,
            "filename": filename,
            "rows": int(len(source_rows)),
            "public": int(source_visibility.get("public", 0)),
            "private": int(source_visibility.get("private", 0)),
            "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
        }
    )
require(
    sum(item["rows"] for item in input_sources) == 323,
    "normalized source row count drifted",
)
require(
    sum(item["public"] for item in input_sources) == 239,
    "normalized public row count drifted",
)
require(
    sum(item["private"] for item in input_sources) == 84,
    "normalized private row count drifted",
)
require(
    input_sources == live_input_sources,
    "normalized source counts or bytes drifted after manifest verification",
)

print(
    f"Validated {len(inventory)} unique repositories across "
    f"{len(SOURCE_FILES)} audit segments; every source total reconciles."
)

Validated 323 unique repositories across 7 audit segments; every source total reconciles.


## Results

In [3]:
visibility_counts = inventory["visibility"].value_counts().to_dict()
class_visibility = pd.crosstab(
    inventory["documentation_class"], inventory["visibility"], margins=True
).reindex(list("ABCDEF") + ["All"], fill_value=0)
dimension_summary = (
    inventory.groupby("visibility")[list(DIMENSIONS)]
    .mean()
    .round(2)
    .T
)

display(Markdown(
    f"**Coverage:** {len(inventory)} repositories — "
    f"{visibility_counts.get('public', 0)} public and "
    f"{visibility_counts.get('private', 0)} private."
))
display(class_visibility)
display(Markdown("### Mean score by visibility (0–4)"))
display(dimension_summary)

**Coverage:** 323 repositories — 239 public and 84 private.

visibility,private,public,All
documentation_class,,,
A,9,36,45
B,14,41,55
C,16,45,61
D,0,12,12
E,6,22,28
F,39,83,122
All,84,239,323


### Mean score by visibility (0–4)

visibility,private,public
orientation,2.24,2.71
technical_depth,2.11,2.57
conceptual_depth,1.94,2.14
commercial_relevance,1.01,1.54
evidence,2.15,2.10
seo_surface,0.29,2.34
cross_linking,1.64,1.90


In [4]:
integrity_terms = (
    "unsupported", "stale", "broken", "mismatch", "contradict",
    "false", "drift", "zero-byte", "ambiguous", "truth repair",
)
inventory["integrity_attention"] = inventory["finding"].str.lower().map(
    lambda text: any(term in text for term in integrity_terms)
)
public_inventory = inventory[inventory["visibility"] == "public"].copy()

private_full_identifiers = set(
    inventory.loc[inventory["visibility"] == "private", "repository"]
)
public_full_identifiers = set(
    inventory.loc[inventory["visibility"] == "public", "repository"]
)
private_slugs = {
    repository.split("/", 1)[-1] for repository in private_full_identifiers
}
public_slug_keys = {
    repository.split("/", 1)[-1].casefold()
    for repository in public_full_identifiers
}
private_only_slugs = {
    slug for slug in private_slugs if slug.casefold() not in public_slug_keys
}
require(
    not {
        repository.casefold() for repository in private_full_identifiers
    } & {
        repository.casefold() for repository in public_full_identifiers
    },
    "a repository identifier is both public and private",
)


def bounded_identifier_pattern(identifiers):
    repository_character = r"A-Za-z0-9._-"
    if not identifiers:
        return r"(?!)"
    alternatives = "|".join(
        re.escape(identifier)
        for identifier in sorted(identifiers, key=len, reverse=True)
    )
    return (
        rf"(?<![{repository_character}])(?:{alternatives})"
        rf"(?:(?=\.git(?:$|[^{repository_character}]))|"
        rf"(?![{repository_character}]))"
    )


private_reference_pattern = re.compile(
    rf"(?P<private_full>{bounded_identifier_pattern(private_full_identifiers)})"
    rf"|(?P<public_full>{bounded_identifier_pattern(public_full_identifiers)})"
    rf"|(?P<private_slug>{bounded_identifier_pattern(private_only_slugs)})",
    flags=re.IGNORECASE,
)

public_prose_rewrites = (
    ("current personal profile/portfolio", "current individual profile/portfolio"),
    ("current personal profile", "current individual profile"),
    ("the personal profile", "the individual profile"),
    ("personal information management", "individual information management"),
)
public_exact_rewrites = {"contrib": "contribution"}


def redact_private_references(value):
    if isinstance(value, str):
        value = public_exact_rewrites.get(value, value)
        for original, replacement in public_prose_rewrites:
            value = value.replace(original, replacement)
        return private_reference_pattern.sub(
            lambda match: (
                match.group(0)
                if match.lastgroup == "public_full"
                else "[private repository]"
            ),
            value,
        )
    if isinstance(value, list):
        return [redact_private_references(item) for item in value]
    if isinstance(value, dict):
        return {
            key: redact_private_references(item) for key, item in value.items()
        }
    return value


for column in public_inventory.columns:
    public_inventory[column] = public_inventory[column].map(
        redact_private_references
    )
public_integrity_count = int(public_inventory["integrity_attention"].sum())

status_rows = pd.DataFrame(
    [
        {"finding": "Public repositories needing explicit truth/identity attention", "count": public_integrity_count},
        {"finding": "Public archived surfaces", "count": int((public_inventory["archived"] == True).sum())},
        {"finding": "Known public READMEs absent in segments that recorded presence", "count": int(((public_inventory["has_readme"] == False)).sum())},
        {"finding": "Public Class A/B/E projects", "count": int(public_inventory["documentation_class"].isin(["A", "B", "E"]).sum())},
    ]
)
display(status_rows)

,finding,count
0,Public repositories needing explicit truth/identity attention,13
1,Public archived surfaces,56
2,Known public READMEs absent in segments that recorded presence,15
3,Public Class A/B/E projects,99


## Authority architecture

The audit rejects a new standalone documentation-engine repository. Existing
authorities already divide the work cleanly:

| Authority | Repository | Owns |
|---|---|---|
| Editorial contract | `organvm/editorial-standards` | README v2, audience templates, rubric, language |
| Data contract | `organvm-iv-taxis/schema-definitions` | `project-record.v1` and assertion/evidence schemas |
| Runtime | `organvm/organvm-engine` | audit and validation commands, machine-readable receipts |
| Fleet policy | `organvm/.github` | organization-wide adoption requirements and repository hygiene |
| Reusable enforcement | `organvm-iv-taxis/system-governance-framework` | reusable CI invoking the runtime |
| Fleet rollout | `organvm-iv-taxis/orchestration-start-here` | conversion waves, execution tracking, compliance reporting |

Public renderers remain downstream consumers: portfolio, showcase, and
stakeholder portal express the same facts for different readers.

## Conversion program

In [5]:
pilots = [
    (1, "4444J99/limen", "claims-ledger and infrastructure/governance pilot"),
    (2, "organvm/the-thing-without-a-name", "creative, scholarly, production, and provenance pilot"),
    (3, "4444J99/peer-audited--behavioral-blockchain", "technical, commercial, ethical, and live-status pilot"),
    (4, "4444J99/your-fit-tailored", "specification-first theory/product pilot"),
    (5, "4444J99/hokage-chess", "prototype-vs-product pilot after mandatory truth repair"),
]

next_twenty = [
    (1, "organvm-iii-ergon/public-record-data-scrapper", "strong public product/evidence front door"),
    (2, "organvm-iii-ergon/life-my--midst--in", "employment, identity theory, architecture, and evaluation"),
    (3, "organvm-iii-ergon/the-actual-news", "public purpose plus protocol/conformance evidence"),
    (4, "organvm-iii-ergon/classroom-rpg-aetheria", "pedagogy, product, system, and evaluator routes"),
    (5, "organvm-iii-ergon/parlor-games--ephemera-engine", "runnable creative system with honest design status"),
    (6, "organvm-ii-poiesis/ivi374ivi027-05", "bounded multimedia/literary work and preservation record"),
    (7, "organvm/a-organvm", "canonical flagship with no audience routing"),
    (8, "organvm/organvm-engine", "largest implementation-to-documentation mismatch"),
    (9, "organvm/radix-recursiva-solve-coagula-redi", "flagship theory/runtime synthesis"),
    (10, "organvm/recursive-engine--generative-entity", "mature theory/runtime flagship"),
    (11, "organvm/linguistic-atomization-framework", "balanced technical-humanities record"),
    (12, "organvm/alchemical-synthesizer", "creative technology flagship"),
    (13, "organvm/metasystem-master", "repair zero-byte and evidence drift before conversion"),
    (14, "organvm/adaptive-personal-syllabus", "repair tracked artifacts and provenance before conversion"),
    (15, "organvm-iv-taxis/system-governance-framework", "document the reusable enforcement surface"),
    (16, "organvm-iv-taxis/distribution-strategy", "source audience and search-intent mappings"),
    (17, "organvm-vii-kerygma/portfolio", "general, hiring, and client renderer"),
    (18, "organvm-vii-kerygma/showcase-portfolio", "humanities, curatorial, and grant renderer"),
    (19, "organvm-vii-kerygma/stakeholder-portal", "evaluator and operational evidence renderer"),
    (20, "organvm/4444J99", "problem-domain top-of-funnel above the internal ontology"),
]

require(
    [rank for rank, _, _ in pilots] == list(range(1, 6)),
    "pilot ranks must be contiguous from 1 through 5",
)
require(
    [rank for rank, _, _ in next_twenty] == list(range(1, 21)),
    "next-twenty ranks must be contiguous from 1 through 20",
)
curated_sequence = [repo for _, repo, _ in pilots + next_twenty]
require(
    len(curated_sequence) == len(set(curated_sequence)),
    "curated queues overlap",
)
curated_names = set(curated_sequence)
public_names = set(public_inventory["repository"])
missing_curated = sorted(curated_names - public_names)
require(
    not missing_curated,
    f"curated public repositories absent from inventory: {missing_curated}",
)

public_by_repository = public_inventory.set_index("repository")
require(
    public_by_repository.index.is_unique,
    "public repository index is not unique",
)


def selection_records(entries):
    selected = []
    for rank, repository, reason in entries:
        source_row = public_by_repository.loc[repository]
        require(
            source_row["finding"] or source_row["priority"] != "unranked",
            f"{repository}: curated entry lacks a finding or priority",
        )
        selected.append(
            {
                "rank": rank,
                "repository": repository,
                "class": source_row["documentation_class"],
                "audit_score": int(source_row["total"]),
                "source_priority": source_row["priority"],
                "integrity_gate": bool(source_row["integrity_attention"]),
                "reason": reason,
            }
        )
    return selected


pilot_records = selection_records(pilots)
queue_records = selection_records(next_twenty)
pilot_table = pd.DataFrame(pilot_records)
queue_table = pd.DataFrame(queue_records)
display(Markdown("### Five pilots"))
display(pilot_table)
display(Markdown("### Next twenty conversions"))
display(queue_table)

### Five pilots

,rank,repository,class,audit_score,source_priority,integrity_gate,reason
0,1,4444J99/limen,A,21,pilot,False,claims-ledger and infrastructure/governance pilot
1,2,organvm/the-thing-without-a-name,A,23,P1,False,"creative, scholarly, production, and provenance pilot"
2,3,4444J99/peer-audited--behavioral-blockchain,A,22,pilot,False,"technical, commercial, ethical, and live-status pilot"
3,4,4444J99/your-fit-tailored,A,24,pilot,False,specification-first theory/product pilot
4,5,4444J99/hokage-chess,B,19,pilot-after-truth-repair,True,prototype-vs-product pilot after mandatory truth repair


### Next twenty conversions

,rank,repository,class,audit_score,source_priority,integrity_gate,reason
0,1,organvm-iii-ergon/public-record-data-scrapper,A,25,ergon_wave_1,False,strong public product/evidence front door
1,2,organvm-iii-ergon/life-my--midst--in,A,26,ergon_wave_1,False,"employment, identity theory, architecture, and evaluation"
2,3,organvm-iii-ergon/the-actual-news,A,26,ergon_wave_1,False,public purpose plus protocol/conformance evidence
3,4,organvm-iii-ergon/classroom-rpg-aetheria,A,25,ergon_wave_1,False,"pedagogy, product, system, and evaluator routes"
4,5,organvm-iii-ergon/parlor-games--ephemera-engine,E,21,ergon_wave_1,False,runnable creative system with honest design status
5,6,organvm-ii-poiesis/ivi374ivi027-05,B,15,unranked,False,bounded multimedia/literary work and preservation record
6,7,organvm/a-organvm,A,18,P0,False,canonical flagship with no audience routing
7,8,organvm/organvm-engine,A,11,P0,True,largest implementation-to-documentation mismatch
8,9,organvm/radix-recursiva-solve-coagula-redi,A,22,P1,False,flagship theory/runtime synthesis
9,10,organvm/recursive-engine--generative-entity,A,23,P1,False,mature theory/runtime flagship


## Truth and authority gates

Conversion is blocked when it would amplify a false current state, broken
canonical identity, unsupported metric, ambiguous license, or placeholder
surface. The initial gate set includes:

- repair Hokage Chess's Python/CLI claims against its bounded TypeScript helper prototype;
- route the 38-commit-behind account-level copy of The Thing to the canonical
  `organvm/the-thing-without-a-name` record instead of forking full editions;
- replace stale Styx test totals with exact-commit receipts and repair its link;
- label Your Fit as specification-first until runtime evidence exists;
- resolve Limen's missing license before calling it open source;
- reconcile organization profiles and transferred-repository identities;
- repair or redirect `organvm/docs`, `organvm/metasystem-master`, and retired Pages copies;
- align editorial, schema, and engine owner URLs with their live authorities.

These are evidence repairs, not copy edits. Search expansion follows the repair.

## Publish privacy-safe artifacts

In [6]:
generated_at = pinned_input_manifest["generated_at"]
input_manifest = pinned_input_manifest
aggregate_summary = {
    "audit_date": "2026-08-31",
    "generated_at": generated_at,
    "scope": {
        "repositories": int(len(inventory)),
        "public": int(visibility_counts.get("public", 0)),
        "private": int(visibility_counts.get("private", 0)),
        "source_segments": source_counts,
        "source_manifest": "reader-mode-input-manifest.json",
        "linked_accounts": 11,
        "coverage_note": "The installation endpoint capped organvm at 100 rows; authenticated repository search added 70 omitted rows. The linked a-organvm account exposed zero repositories; organvm/a-organvm is a distinct repository.",
    },
    "documentation_classes": {
        visibility: {
            cls: int(class_visibility.loc[cls].get(visibility, 0))
            for cls in "ABCDEF"
        }
        for visibility in ("public", "private")
    },
    "documentation_class_totals": {
        cls: int(class_visibility.loc[cls].get("All", 0))
        for cls in "ABCDEF"
    },
    "mean_scores": {
        visibility: {
            dimension: round(float(value), 3)
            for dimension, value in group[list(DIMENSIONS)].mean().items()
        }
        for visibility, group in inventory.groupby("visibility")
    },
    "public_integrity_attention": public_integrity_count,
    "privacy": "Private repository names and row-level findings are excluded from committed artifacts.",
}

public_columns = [
    "repository", "visibility", "documentation_class", "repository_role",
    "priority", "archived", "has_readme", *DIMENSIONS, "source_total", "total",
    "integrity_attention", "finding", "recommended_treatment", "url",
    "recommended_reader_modes", "industry_clusters", "concept_clusters", "topics",
]
public_records = json.loads(
    public_inventory[public_columns].to_json(orient="records")
)
rollout = {
    "audit_date": "2026-08-31",
    "generated_at": generated_at,
    "dimensions": list(DIMENSIONS),
    "authority_repositories": [
        "organvm/editorial-standards",
        "organvm-iv-taxis/schema-definitions",
        "organvm/organvm-engine",
        "organvm/.github",
        "organvm-iv-taxis/system-governance-framework",
        "organvm-iv-taxis/orchestration-start-here",
    ],
    "selection_method": (
        "Editorial order combining class, public leverage, source-segment "
        "recommendations, and truth/identity gates; audit_score is descriptive, "
        "not the sole ranking function."
    ),
    "pilots": pilot_records,
    "next_twenty": queue_records,
    "public_inventory": public_records,
}

class_rows = [
    f"| {cls} | {int(class_visibility.loc[cls].get('public', 0))} | "
    f"{int(class_visibility.loc[cls].get('private', 0))} |"
    for cls in "ABCDEF"
]
pilot_rows = [
    f"| {item['rank']} | `{item['repository']}` | {item['class']} | "
    f"{item['audit_score']} | {item['source_priority']} | {item['reason']} |"
    for item in pilot_records
]
queue_rows = [
    f"| {item['rank']} | `{item['repository']}` | {item['class']} | "
    f"{item['audit_score']} | {item['source_priority']} | {item['reason']} |"
    for item in queue_records
]
report_lines = [
    "# ORGANVM reader-mode documentation estate audit",
    "",
    "Audit date: 2026-08-31",
    "",
    "## Outcome",
    "",
    (
        f"All **{len(inventory)}** repositories visible through the linked "
        f"GitHub context were classified and scored: "
        f"**{visibility_counts.get('public', 0)} public** and "
        f"**{visibility_counts.get('private', 0)} private**. The implementation "
        "uses existing editorial, schema, runtime, governance, and orchestration "
        "authorities rather than adding a documentation-engine repository."
    ),
    "",
    (
        "The public contract is one canonical factual project record, "
        "evidence-linked material claims, class-specific reader routes, and "
        "truth repair before search expansion. Private repository identifiers "
        "and row-level findings remain outside this report."
    ),
    "",
    "## Data and validation",
    "",
    (
        "The audit uses seven authorized exports at one "
        "repository per row. All 323 source totals reconcile with the seven "
        "0–4 dimension scores; repository identifiers are unique; and class, "
        "visibility, and expected coverage checks pass."
    ),
    "",
    (
        "The public input manifest records source filenames, row and visibility "
        "counts, and SHA-256 hashes. Raw exports are not committed, so rerunning "
        "is possible only with separately retained authorized copies."
    ),
    "",
    "## Documentation classes",
    "",
    "| Class | Public | Private |",
    "|---|---:|---:|",
    *class_rows,
    "",
    "## Authority map",
    "",
    "| Responsibility | Canonical repository |",
    "|---|---|",
    "| Editorial contract, rubric, and templates | `organvm/editorial-standards` |",
    "| Project and assertion schemas | `organvm-iv-taxis/schema-definitions` |",
    "| Audit and validation runtime | `organvm/organvm-engine` |",
    "| Fleet adoption policy | `organvm/.github` |",
    "| Reusable CI invocation | `organvm-iv-taxis/system-governance-framework` |",
    "| Rollout waves and execution receipts | `organvm-iv-taxis/orchestration-start-here` |",
    "",
    "## Five pilots",
    "",
    (
        "Ranks combine public leverage, rhetorical coverage, source recommendations, "
        "and truth gates. The audit score is shown for traceability, not used as a "
        "standalone ordering function."
    ),
    "",
    "| Rank | Repository | Class | Score / 28 | Source priority | Why this pilot |",
    "|---:|---|:---:|---:|---|---|",
    *pilot_rows,
    "",
    "## Next twenty conversions",
    "",
    "| Rank | Repository | Class | Score / 28 | Source priority | Value / gap / leverage |",
    "|---:|---|:---:|---:|---|---|",
    *queue_rows,
    "",
    "## Gates before expansion",
    "",
    "- Correct false or stale implementation, deployment, test, owner, license, and provenance claims.",
    "- Do not create audience editions for archives, forks, or deployment artifacts beyond their class minimum.",
    "- Treat industry mappings as proposed unless a claim record substantiates pilot or deployment status.",
    "- Keep private repository identifiers and evidence out of public generated inventories.",
    "- Preserve the endpoint coverage caveat recorded in the aggregate summary.",
    "",
    "## Reproduction boundary",
    "",
    (
        "Run `build_reader_mode_estate_audit.py` with "
        "`ORGANVM_DOC_AUDIT_INPUT_DIR` pointing to separately retained, "
        "authorized copies of the seven exports whose hashes appear in "
        "`reader-mode-input-manifest.json`. The public repository alone is "
        "not a self-contained copy of the raw audit data."
    ),
]
report = "\n".join(report_lines) + "\n"

serialized_artifacts = {
    "reader-mode-input-manifest.json": json.dumps(
        input_manifest, indent=2, allow_nan=False
    ) + "\n",
    "reader-mode-estate-summary.json": json.dumps(
        aggregate_summary, indent=2, allow_nan=False
    ) + "\n",
    "reader-mode-public-rollout.json": json.dumps(
        rollout, indent=2, allow_nan=False
    ) + "\n",
    "reader-mode-estate-audit.md": report,
}
manifest_scan_view = json.loads(json.dumps(input_manifest))
for source in manifest_scan_view["sources"]:
    source["source_segment"] = "[source segment]"
summary_scan_view = json.loads(json.dumps(aggregate_summary))
summary_scan_view["scope"]["source_segments"] = list(
    summary_scan_view["scope"]["source_segments"].values()
)
privacy_scan_artifacts = {
    **serialized_artifacts,
    "reader-mode-input-manifest.json": json.dumps(manifest_scan_view),
    "reader-mode-estate-summary.json": json.dumps(summary_scan_view),
}
privacy_hits = {
    filename: sum(
        match.lastgroup != "public_full"
        for match in private_reference_pattern.finditer(payload)
    )
    for filename, payload in privacy_scan_artifacts.items()
}
require(
    not any(privacy_hits.values()),
    "private repository identifiers detected in public payloads: "
    f"{sum(privacy_hits.values())} hit(s)",
)

for filename, payload in serialized_artifacts.items():
    (OUTPUT_DIR / filename).write_text(payload, encoding="utf-8")

print("Wrote reader-mode-input-manifest.json")
print("Wrote reader-mode-estate-summary.json")
print("Wrote reader-mode-public-rollout.json")
print("Wrote reader-mode-estate-audit.md")

Wrote reader-mode-input-manifest.json
Wrote reader-mode-estate-summary.json
Wrote reader-mode-public-rollout.json
Wrote reader-mode-estate-audit.md


## Takeaways

1. Documentation density is not the estate's primary failure; shared entry
   order and authority drift are.
2. Class A projects warrant five routes, while components, deployment
   artifacts, research works, and archives should retain smaller contracts.
3. The first implementation wave must couple rhetoric with claim-level
   evidence, otherwise SEO multiplies contradictions.
4. The five pilots intentionally cover infrastructure, creative practice,
   technical/commercial systems, specification-first work, and design-only
   truth repair.
5. The public rollout can be generated without exposing private repository
   identities.